In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)

In [2]:
PROJECT_DIR = "/Users/hoyin/Desktop/DanaFarber/workspaces/CMI_Painter_Angiosarcoma_WES_analysis_mh_regional/scripts"
RAW_DATA_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DATA_DIR = f"{PROJECT_DIR}/data/processed"


## General formatting

In [25]:
treatment_ehr_df = pd.read_excel(f"{RAW_DATA_DIR}/Batches_1-12_Dates_Notes_Masked (4).xlsx",sheet_name="TX")
treatment_ehr_df["STUDY ID"] = treatment_ehr_df["STUDY ID"].apply(lambda x: x.replace("ASCPROJECT","ASCProject"))
treatment_ehr_df.columns = [col.strip() for col in treatment_ehr_df.columns]
#treatment_ehr_df = treatment_ehr_df.replace({"NOT FOUND IN RECORD":np.NaN,"NA": np.NaN})
#treatment_ehr_df = treatment_ehr_df.dropna(subset=["DRUG","MODE"])
treatment_ehr_df["DRUG"] = treatment_ehr_df["DRUG"].str.strip()
treatment_ehr_df = treatment_ehr_df.sort_values(by=["STUDY ID","START DATE (DAYS FROM DX)"])
treatment_ehr_df = treatment_ehr_df.loc[:, ~treatment_ehr_df.columns.str.contains('^Unnamed')]
#treatment_ehr_df = treatment_ehr_df[~treatment_ehr_df["MODE"].str.contains("OTHER")]
treatment_ehr_df


,STUDY ID,START DATE (DAYS FROM DX),STOP DATE (DAYS FROM DX),DRUG,MODE,CLINICAL TRIAL,TX RESPONSE,D/C REASON,ADVERSE EFFECTS
196,ASCProject_0003,106,264,IFOSFAMIDE,ADJUVANT,NO,RESPONSE,FINISHED COURSE,"FACIAL DROOPINESS, NUMBNESS, TONGUE NUMBNESS, ..."
197,ASCProject_0003,106,264,DOXORUBICIN,ADJUVANT,NO,RESPONSE,FINISHED COURSE,"FACIAL DROOPINESS, NUMBNESS, TONGUE NUMBNESS, ..."
198,ASCProject_0003,106,264,MESNA,ADJUVANT,NO,NaN,NOT FOUND IN RECORD,NOT FOUND IN RECORD
199,ASCProject_0003,106,264,DEXRAZOXANE,ADJUVANT,NO,NaN,NOT FOUND IN RECORD,NOT FOUND IN RECORD
200,ASCProject_0003,264,698,DOCETAXEL,ADJUVANT,NO,STABLE,ADVERSE EFFECTS,"HOT FLASHES, THROMBOCYTOPENIA, NEUROPATHY"
...,...,...,...,...,...,...,...,...,...
592,ASCProject_PZCXCF,49,49,PACLITAXEL,METS,NO,NOT FOUND IN RECORD,ON TX AT LAST ENCOUNTER,NOT FOUND IN RECORD
440,ASCProject_PZUDLM,-1031,-940,DOXORUBICIN,OTHER CANCER,NO,NOT FOUND IN RECORD,FINISHED COURSE,NOT FOUND IN RECORD
441,ASCProject_PZUDLM,-1031,-940,CYCLOPHOSPHAMIDE,OTHER CANCER,NO,NOT FOUND IN RECORD,FINISHED COURSE,NOT FOUND IN RECORD
442,ASCProject_PZUDLM,-940,-848,PACLITAXEL,OTHER CANCER,NO,NOT FOUND IN RECORD,FINISHED COURSE,"BONE PAIN, PERIPHERAL NEUROPATHY"


## Replace some values in MODE (formatted)

In [45]:
treatment_ehr_df["MODE (FORMATTED)"] = treatment_ehr_df["MODE"]
treatment_ehr_df.loc[treatment_ehr_df["MODE"].str.contains("OTHER")==True,"MODE (FORMATTED)"] = "OTHER"
treatment_ehr_df.loc[
    treatment_ehr_df["MODE"].str.contains("N/A|NONE|NOT FOUND IN RECORD")==True,
    "MODE (FORMATTED)"
] = "UNKNOWN"
treatment_ehr_df["MODE (FORMATTED)"] = treatment_ehr_df["MODE (FORMATTED)"].fillna("UNKNOWN")
treatment_ehr_df["MODE (FORMATTED)"] = treatment_ehr_df["MODE (FORMATTED)"].replace({"","UNKNOWN"})


treatment_ehr_df#.dropna(axis=,#subset=[""])

,STUDY ID,START DATE (DAYS FROM DX),STOP DATE (DAYS FROM DX),DRUG,MODE,CLINICAL TRIAL,TX RESPONSE,D/C REASON,ADVERSE EFFECTS,MODE (FORMATTED)
196,ASCProject_0003,106,264,IFOSFAMIDE,ADJUVANT,NO,RESPONSE,FINISHED COURSE,"FACIAL DROOPINESS, NUMBNESS, TONGUE NUMBNESS, ...",ADJUVANT
197,ASCProject_0003,106,264,DOXORUBICIN,ADJUVANT,NO,RESPONSE,FINISHED COURSE,"FACIAL DROOPINESS, NUMBNESS, TONGUE NUMBNESS, ...",ADJUVANT
198,ASCProject_0003,106,264,MESNA,ADJUVANT,NO,NaN,NOT FOUND IN RECORD,NOT FOUND IN RECORD,ADJUVANT
199,ASCProject_0003,106,264,DEXRAZOXANE,ADJUVANT,NO,NaN,NOT FOUND IN RECORD,NOT FOUND IN RECORD,ADJUVANT
200,ASCProject_0003,264,698,DOCETAXEL,ADJUVANT,NO,STABLE,ADVERSE EFFECTS,"HOT FLASHES, THROMBOCYTOPENIA, NEUROPATHY",ADJUVANT
...,...,...,...,...,...,...,...,...,...,...
592,ASCProject_PZCXCF,49,49,PACLITAXEL,METS,NO,NOT FOUND IN RECORD,ON TX AT LAST ENCOUNTER,NOT FOUND IN RECORD,METS
440,ASCProject_PZUDLM,-1031,-940,DOXORUBICIN,OTHER CANCER,NO,NOT FOUND IN RECORD,FINISHED COURSE,NOT FOUND IN RECORD,OTHER
441,ASCProject_PZUDLM,-1031,-940,CYCLOPHOSPHAMIDE,OTHER CANCER,NO,NOT FOUND IN RECORD,FINISHED COURSE,NOT FOUND IN RECORD,OTHER
442,ASCProject_PZUDLM,-940,-848,PACLITAXEL,OTHER CANCER,NO,NOT FOUND IN RECORD,FINISHED COURSE,"BONE PAIN, PERIPHERAL NEUROPATHY",OTHER


In [46]:
treatment_ehr_df.to_csv(f"{PROCESSED_DATA_DIR}/treatment_data.tsv",sep="\t",index=False)

In [47]:
treatment_ehr_df["MODE (FORMATTED)"].value_counts()

METS                278
ADJUVANT            145
OTHER               106
NEOADJUVANT          91
LOCAL RECURRENCE     43
UNKNOWN              36
Name: MODE (FORMATTED), dtype: int64